# Experiment 5.3.2.1 — RSNN WHEN objective upper-bound sweep

Analysis-only notebook. The frozen WHAT trajectory and `rsnn_shortmem` architecture are fixed; only the supervised WHEN objective changes.

Primary cross-objective comparisons use the same post-hoc probes on frozen membrane state `U_t`, because `phase_only` and `progress_only` intentionally leave one native head untrained.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not locate writingRing repository root")

ROOT = find_repo_root()
OUT = ROOT / "notebooks/artifacts/experiment_5_3_2_1_when_objective_sweep/rsnn_when_objective_v1"

manifest = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))
runs = pd.read_csv(OUT / "runs.csv")
histories = pd.read_csv(OUT / "histories.csv")
probe_runs = pd.read_csv(OUT / "probe_runs.csv")
ablation_runs = pd.read_csv(OUT / "ablation_runs.csv")
history_gain_runs = pd.read_csv(OUT / "history_gain_runs.csv")
gradient_diagnostics = pd.read_csv(OUT / "gradient_diagnostics.csv")
baseline_runs = pd.read_csv(OUT / "baseline_runs.csv")
local_reference = pd.read_csv(OUT / "local_reference.csv")

manifest


## 1. Primary WHEN representation quality

Compare objectives with the common post-hoc `U_t` probes. Lower progress MAE, higher phase BA, higher per-gesture Spearman, and lower monotonic violation are better.


In [ ]:
objective_order = ["phase_only", "phase_heavy", "joint", "progress_heavy", "progress_only"]

primary = (
    runs.groupby("objective", as_index=False)
    .agg(
        phase_ba_mean=("probe_u_test_phase_ba", "mean"),
        phase_ba_sd=("probe_u_test_phase_ba", "std"),
        progress_mae_mean=("probe_test_sample_balanced_mae_mean", "mean"),
        progress_mae_sd=("probe_test_sample_balanced_mae_mean", "std"),
        spearman_mean=("probe_test_spearman_mean", "mean"),
        spearman_sd=("probe_test_spearman_mean", "std"),
        monotonic_violation_mean=("probe_test_monotonic_violation_rate_mean", "mean"),
        monotonic_violation_sd=("probe_test_monotonic_violation_rate_mean", "std"),
    )
)
primary["objective"] = pd.Categorical(primary["objective"], objective_order, ordered=True)
primary = primary.sort_values("objective")
primary


## 2. Elapsed-time and WHAT-only baselines

`elapsed_time_only` receives only `t/fs`, never the final duration `T`. It is the clock baseline that every learned WHEN representation should be judged against.


In [ ]:
baseline_summary = (
    baseline_runs.groupby("baseline", as_index=False)
    .agg(
        phase_ba_mean=("phase_probe_test_ba", "mean"),
        phase_ba_sd=("phase_probe_test_ba", "std"),
        progress_mae_mean=("progress_probe_test_sample_balanced_mae_mean", "mean"),
        progress_mae_sd=("progress_probe_test_sample_balanced_mae_mean", "std"),
        spearman_mean=("progress_probe_test_spearman_mean", "mean"),
        monotonic_violation_mean=("progress_probe_test_monotonic_violation_rate_mean", "mean"),
    )
)
baseline_summary


In [ ]:
elapsed = baseline_summary.loc[baseline_summary["baseline"].eq("elapsed_time_only")].iloc[0]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for _, row in primary.iterrows():
    ax.scatter(row["progress_mae_mean"], row["phase_ba_mean"], s=70)
    ax.annotate(str(row["objective"]), (row["progress_mae_mean"], row["phase_ba_mean"]), xytext=(5, 5), textcoords="offset points")

ax.scatter(elapsed["progress_mae_mean"], elapsed["phase_ba_mean"], marker="x", s=100, label="elapsed-time only")
ax.set_xlabel("Test progress MAE (sample-balanced; lower is better)")
ax.set_ylabel("Test 10-bin phase balanced accuracy")
ax.set_title("Exp5.3.2.1: WHEN objective trade-off")
ax.legend()
plt.show()


## 3. Temporal trajectory quality

A useful progress representation should not merely have low pointwise MAE; its decoded trajectory should also move forward consistently within each gesture.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(primary))
ax.bar(x, primary["spearman_mean"])
ax.set_xticks(x, primary["objective"].astype(str), rotation=25, ha="right")
ax.set_ylabel("Mean per-gesture Spearman")
ax.set_title("Decoded progress ordering quality")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(x, primary["monotonic_violation_mean"])
ax.set_xticks(x, primary["objective"].astype(str), rotation=25, ha="right")
ax.set_ylabel("Mean monotonic violation rate")
ax.set_title("Decoded progress backward-step rate")
plt.show()


## 4. Causal history attribution

For phase, positive `H_reset` / `H_shuffle` means ordered history improves phase accessibility.

For progress MAE, positive history gain means removing history increases error.


In [ ]:
history_summary = (
    history_gain_runs.groupby("objective", as_index=False)
    .agg(
        H_reset_phase_mean=("H_reset_phase_ba", "mean"),
        H_reset_phase_sd=("H_reset_phase_ba", "std"),
        H_shuffle_phase_mean=("H_shuffle_phase_ba", "mean"),
        H_shuffle_phase_sd=("H_shuffle_phase_ba", "std"),
        H_reset_progress_mean=("H_reset_progress_mae", "mean"),
        H_reset_progress_sd=("H_reset_progress_mae", "std"),
        H_shuffle_progress_mean=("H_shuffle_progress_mae", "mean"),
        H_shuffle_progress_sd=("H_shuffle_progress_mae", "std"),
        H_reset_spearman_mean=("H_reset_spearman", "mean"),
        H_shuffle_spearman_mean=("H_shuffle_spearman", "mean"),
    )
)
history_summary["objective"] = pd.Categorical(history_summary["objective"], objective_order, ordered=True)
history_summary = history_summary.sort_values("objective")
history_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(history_summary))
width = 0.36
ax.bar(x - width/2, history_summary["H_reset_phase_mean"], width, label="H_reset")
ax.bar(x + width/2, history_summary["H_shuffle_phase_mean"], width, label="H_shuffle")
ax.axhline(0.0, linewidth=1)
ax.set_xticks(x, history_summary["objective"].astype(str), rotation=25, ha="right")
ax.set_ylabel("Phase BA gain from ordered history")
ax.set_title("Does the objective preserve order-dependent WHEN?")
ax.legend()
plt.show()


## 5. Gradient dominance

The raw loss values have very different scales. These diagnostics show what fraction of shared-RSNN gradient magnitude comes from phase vs progress after applying each objective weight.


In [ ]:
gradient_summary = (
    gradient_diagnostics.groupby("objective", as_index=False)
    .agg(
        phase_fraction_mean=("mean_weighted_phase_grad_fraction", "mean"),
        phase_fraction_sd=("mean_weighted_phase_grad_fraction", "std"),
        progress_fraction_mean=("mean_weighted_progress_grad_fraction", "mean"),
        progress_fraction_sd=("mean_weighted_progress_grad_fraction", "std"),
        raw_phase_grad_mean=("mean_raw_phase_grad_norm", "mean"),
        raw_progress_grad_mean=("mean_raw_progress_grad_norm", "mean"),
    )
)
gradient_summary["objective"] = pd.Categorical(gradient_summary["objective"], objective_order, ordered=True)
gradient_summary = gradient_summary.sort_values("objective")
gradient_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(gradient_summary))
ax.bar(x, gradient_summary["phase_fraction_mean"], label="phase contribution")
ax.bar(x, gradient_summary["progress_fraction_mean"], bottom=gradient_summary["phase_fraction_mean"], label="progress contribution")
ax.set_xticks(x, gradient_summary["objective"].astype(str), rotation=25, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Mean weighted gradient fraction")
ax.set_title("Shared RSNN gradient dominance")
ax.legend()
plt.show()


## 6. Spike accessibility

The final system is intended to remain SNN-friendly. Compare how much WHEN remains accessible from membrane/synaptic state, instantaneous spikes, and trailing spike-count windows.


In [ ]:
access = (
    probe_runs.groupby(["objective", "feature_type"], as_index=False)
    .agg(
        phase_ba_mean=("phase_probe_test_ba", "mean"),
        progress_mae_mean=("progress_probe_test_mae", "mean"),
    )
)
phase_pivot = access.pivot(index="objective", columns="feature_type", values="phase_ba_mean").reindex(objective_order)
progress_pivot = access.pivot(index="objective", columns="feature_type", values="progress_mae_mean").reindex(objective_order)

phase_pivot


In [ ]:
progress_pivot


## 7. Validation learning curves

Check whether an objective improves the shared representation early and then overfits. Checkpoint selection is validation-only and objective-aligned.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for objective in objective_order:
    subset = histories[histories["objective"].eq(objective)]
    summary = subset.groupby("epoch", as_index=False)["val_objective_loss"].mean()
    ax.plot(summary["epoch"], summary["val_objective_loss"], label=objective)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean validation weighted objective")
ax.set_title("Exp5.3.2.1 validation objective")
ax.legend()
plt.show()


## Interpretation

Do **not** select the winner from progress MAE alone.

A strong candidate for the later WHAT x WHEN fusion should ideally:
1. beat the elapsed-time baseline on progress;
2. retain useful phase accessibility;
3. have high per-gesture Spearman and low monotonic violation;
4. keep positive `H_reset` and `H_shuffle`, so the representation depends on real ordered history rather than only clock-like timing;
5. preserve WHEN information in spike-domain readouts when possible.

This experiment estimates the supervised objective ceiling of the fixed 128-neuron `rsnn_shortmem` WHEN branch. Width/depth changes belong in later Exp5.3.2.x ablations.
